# Soal 11.1

Sistem tridiagonal diselesaikan dengan dua cara: algoritma Thomas seperti Example 11.1 dan Gauss-Seidel seperti Example 11.3.

In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

def thomas(a, d, c, b):
    a = np.array(a, dtype=float)
    d = np.array(d, dtype=float)
    c = np.array(c, dtype=float)
    b = np.array(b, dtype=float)
    n = len(d)
    l = np.zeros(n - 1)
    u = d.copy()
    r = b.copy()
    for i in range(1, n):
        l[i - 1] = a[i - 1] / u[i - 1]
        u[i] = u[i] - l[i - 1] * c[i - 1]
        r[i] = r[i] - l[i - 1] * r[i - 1]
    x = np.zeros(n)
    x[-1] = r[-1] / u[-1]
    for i in range(n - 2, -1, -1):
        x[i] = (r[i] - c[i] * x[i + 1]) / u[i]
    L = np.eye(n)
    U = np.zeros((n, n))
    for i in range(n):
        U[i, i] = u[i]
        if i < n - 1:
            L[i + 1, i] = l[i]
            U[i, i + 1] = c[i]
    return L, U, r, x

def gauss_seidel(A, b, x0=None, es=5.0, max_iter=100, lam=1.0):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    x = np.zeros(n) if x0 is None else np.array(x0, dtype=float)
    rows = []
    for it in range(1, max_iter + 1):
        old = x.copy()
        for i in range(n):
            raw = (b[i] - np.dot(A[i, :i], x[:i]) - np.dot(A[i, i + 1:], old[i + 1:])) / A[i, i]
            x[i] = lam * raw + (1 - lam) * old[i]
        ea = np.full(n, np.nan)
        mask = x != 0
        ea[mask] = np.abs((x[mask] - old[mask]) / x[mask]) * 100
        rows.append([it, *x, np.nanmax(ea)])
        if np.nanmax(ea) < es:
            break
    return x, np.array(rows)

def jacobi(A, b, x0=None, es=5.0, max_iter=100):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    x = np.zeros(n) if x0 is None else np.array(x0, dtype=float)
    rows = []
    for it in range(1, max_iter + 1):
        old = x.copy()
        for i in range(n):
            x[i] = (b[i] - np.dot(A[i, :i], old[:i]) - np.dot(A[i, i + 1:], old[i + 1:])) / A[i, i]
        ea = np.full(n, np.nan)
        mask = x != 0
        ea[mask] = np.abs((x[mask] - old[mask]) / x[mask]) * 100
        rows.append([it, *x, np.nanmax(ea)])
        if np.nanmax(ea) < es:
            break
    return x, np.array(rows)

A = np.array([[0.8, -0.4, 0.0], [-0.4, 0.8, -0.4], [0.0, -0.4, 0.8]])
b = np.array([41.0, 25.0, 105.0])
L, U, r, x_thomas = thomas([-0.4, -0.4], [0.8, 0.8, 0.8], [-0.4, -0.4], b)
x_gs, hist = gauss_seidel(A, b, es=5)
print("L=\n", L)
print("U=\n", U)
print("RHS setelah eliminasi =", r)
print("Thomas x =", x_thomas)
print("Gauss-Seidel es<5% x =", x_gs)
print("Iterasi GS =\n", hist)
print("np.linalg.solve =", np.linalg.solve(A, b))

L=
 [[ 1.        0.        0.      ]
 [-0.5       1.        0.      ]
 [ 0.       -0.666667  1.      ]]
U=
 [[ 0.8      -0.4       0.      ]
 [ 0.        0.6      -0.4     ]
 [ 0.        0.        0.533333]]
RHS setelah eliminasi = [ 41.        45.5      135.333333]
Thomas x = [173.75 245.   253.75]
Gauss-Seidel es<5% x = [167.871094 239.121094 250.810547]
Iterasi GS =
 [[  1.        51.25      56.875    159.6875   100.      ]
 [  2.        79.6875   150.9375   206.71875   62.318841]
 [  3.       126.71875  197.96875  230.234375  37.114673]
 [  4.       150.234375 221.484375 241.992188  15.652626]
 [  5.       161.992188 233.242188 247.871094   7.258259]
 [  6.       167.871094 239.121094 250.810547   3.502036]]
np.linalg.solve = [173.75 245.   253.75]


Algoritma Thomas memberi solusi langsung `x = [173.75, 245, 253.75]`. Gauss-Seidel dari tebakan nol berhenti pada iterasi ke-6 untuk `es < 5%`; jika toleransi diperkecil, hasilnya menuju nilai Thomas.